# 🎙️ RepublicaVideo1 · Paso 2: pruebas de voz (Chatterbox multilingüe vs XTTS-v2)
**Corre en CPU** (Entorno de ejecución → Cambiar tipo → **CPU**): no gasta GPU. Al terminar se desconecta solo.

- Mismas 3 frases por personaje con los dos motores, en español.
- **CONFUNDIDO**: aguda, rápida, exagerada · **EXPLICADOR**: grave, plana, tranquila.
- Sin clonar voces reales: Chatterbox toma como referencia voces **sintéticas** (Kokoro); XTTS-v2 usa sus voces de estudio incluidas.
- ⚠ XTTS-v2 tiene licencia **no comercial** (Coqui CPML). Chatterbox es MIT.
- Resultado en `RepublicaVideo1/audios/pruebas/`: cada audio por separado, `duraciones.md` con la duración real y una secuencia por motor con pausas de 0.3 s. El gasto queda en `RepublicaVideo1/gasto_gpu.csv`.

In [ ]:
#@title 1 · Preparar (Drive, control de gasto y voces IA) — corre en CPU, no necesitas GPU
#@markdown **Entorno de ejecución → Cambiar tipo → CPU** (este paso no usa GPU).
CACHE_EN_DRIVE = True  #@param {type:"boolean"}
DESCONECTAR_AL_TERMINAR = True  #@param {type:"boolean"}
import os, sys, time, json, subprocess, csv, datetime
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive; drive.mount('/content/drive')
RV = '/content/drive/MyDrive/RepublicaVideo1' if IN_COLAB else os.path.abspath('RepublicaVideo1')
for d_ in ('audios', 'rig', 'animatic', 'renders', 'final'): os.makedirs(os.path.join(RV, d_), exist_ok=True)
def gpu_name():
    try: return subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() or 'CPU'
    except FileNotFoundError: return 'CPU'
GASTO = os.path.join(RV, 'gasto_gpu.csv')
def gpu_total_min():
    if not os.path.exists(GASTO): return 0.0
    return sum(float(r['min_gpu']) for r in csv.DictReader(open(GASTO)))
class Paso:
    """Mide el tiempo del paso; sólo cuenta como GPU si la máquina tiene GPU."""
    def __init__(self, nombre): self.nombre = nombre
    def __enter__(self):
        self.t0 = time.time(); self.dev = gpu_name()
        print(f"▶ {self.nombre} · máquina: {self.dev} · GPU usada hasta ahora: {gpu_total_min():.1f} min"); return self
    def __exit__(self, et, ev, tb):
        mins = (time.time() - self.t0) / 60; g = mins if self.dev != 'CPU' else 0.0
        new = not os.path.exists(GASTO)
        with open(GASTO, 'a', newline='') as f_:
            w = csv.writer(f_)
            if new: w.writerow(['paso', 'inicio', 'min_total', 'min_gpu', 'maquina', 'ok'])
            w.writerow([self.nombre, datetime.datetime.now().isoformat(timespec='seconds'), f"{mins:.2f}", f"{g:.2f}", self.dev, et is None])
        print(f"■ {self.nombre} · {mins:.1f} min ({g:.1f} de GPU) · GPU total: {gpu_total_min():.1f} min")
        if et is not None: print("✗ Error:", str(ev)[:300]); desconectar()
        return False
def desconectar():
    if IN_COLAB and DESCONECTAR_AL_TERMINAR:
        print("Desconectando el entorno para no gastar unidades…"); from google.colab import runtime; runtime.unassign()
CACHE = '/content/drive/MyDrive/.cache_colab'
def restore_env(name):
    tar = f"{CACHE}/{name}.tar"
    if CACHE_EN_DRIVE and os.path.exists(tar) and not os.path.exists(f"/content/{name}/.ok"):
        subprocess.run(['tar', '-xf', tar, '-C', '/'], capture_output=True)
    return os.path.exists(f"/content/{name}/.ok")
def save_env(name):
    if CACHE_EN_DRIVE and IN_COLAB:
        os.makedirs(CACHE, exist_ok=True); tmp = f"/content/{name}.tar"
        parts = [f"content/{name}"] + (['root/.local/share/uv/python'] if os.path.isdir('/root/.local/share/uv/python') else [])
        subprocess.run(['tar', '-cf', tmp, '-C', '/', *parts]); subprocess.run(['cp', tmp, f"{CACHE}/{name}.tar"]); os.remove(tmp)
with Paso('2a · instalar voces'):
    if IN_COLAB and not restore_env('voz_env'):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
        subprocess.run(['apt-get', '-qq', 'install', '-y', 'espeak-ng'], capture_output=True)
        subprocess.run(['uv', 'venv', '-q', '--python', '3.11', '/content/voz_env'], check=True)
        r = subprocess.run(['uv', 'pip', 'install', '-q', '--python', '/content/voz_env/bin/python', 'chatterbox-tts==0.1.7', 'coqui-tts', 'six',
                            'kokoro>=0.9.4', 'soundfile'], capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError("No se instalaron las voces: " + r.stderr[-500:])
        open('/content/voz_env/.ok', 'w').write('ok'); save_env('voz_env')
    print("Voces listas ✓")

In [ ]:
#@title 2 · Pruebas de voz: las mismas 3 frases por personaje con Chatterbox multilingüe y con XTTS-v2
#@markdown Sin clonar a nadie: Chatterbox usa como referencia voces sintéticas de Kokoro; XTTS-v2 usa sus voces de estudio incluidas.
VOZ_XTTS_CONFUNDIDO = "Andrew Chipper"  #@param {type:"string"}
VOZ_XTTS_EXPLICADOR = "Damien Black"  #@param {type:"string"}
FRASES = {
    'confundido': ["Oye… ¿qué es esto?", "¿Por qué traigo una banda?", "¿¡Qué elecciones!?"],
    'explicador': ["Tu credencial. Eres ciudadano.", "Ganaste las elecciones. Ayer.", "Es una república."],
}
# dirección de voz: CONFUNDIDO aguda, rápida, exagerada · EXPLICADOR grave, plana, tranquila
ESTILO = {'confundido': {'kokoro': 'em_santa', 'exag': 0.85, 'cfg': 0.3, 'temp': 0.9, 'semitonos': 3.0, 'tempo': 1.08, 'xtts_speed': 1.15},
          'explicador': {'kokoro': 'em_alex', 'exag': 0.25, 'cfg': 0.5, 'temp': 0.6, 'semitonos': -2.0, 'tempo': 0.97, 'xtts_speed': 0.95}}
OUT = os.path.join(RV, 'audios', 'pruebas'); os.makedirs(OUT, exist_ok=True); TMP = '/content/voz_tmp' if IN_COLAB else os.path.abspath('voz_tmp'); os.makedirs(TMP, exist_ok=True)
SCRIPT = r'''
import sys, os, json, numpy as np, soundfile as sf, torch
cfg = json.load(open(sys.argv[1])); dev = 'cuda' if torch.cuda.is_available() else 'cpu'; os.environ['COQUI_TOS_AGREED'] = '1'
T = cfg['tmp']; out = []
# 1) referencias sintéticas (Kokoro) para Chatterbox
from kokoro import KPipeline
kp = KPipeline(lang_code='e', repo_id='hexgrad/Kokoro-82M')
refs = {}
for who, st in cfg['estilo'].items():
    a = np.concatenate([np.asarray(x[2], np.float32) for x in kp("Hola. Esta es una muestra de voz para el personaje, con calma y claridad.", voice=st['kokoro'])])
    p = os.path.join(T, f'ref_{who}.wav'); sf.write(p, a, 24000); refs[who] = p
# 2) Chatterbox multilingüe
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
cb = ChatterboxMultilingualTTS.from_pretrained(device=torch.device(dev))
for who, frases in cfg['frases'].items():
    st = cfg['estilo'][who]
    for i, t in enumerate(frases):
        w = cb.generate(t, language_id='es', audio_prompt_path=refs[who], exaggeration=st['exag'], cfg_weight=st['cfg'], temperature=st['temp'])
        p = os.path.join(T, f'chatterbox_{who}_{i + 1}.wav'); sf.write(p, w.squeeze(0).cpu().numpy(), cb.sr); print('OK', p, flush=True)
del cb
# 3) XTTS-v2 con voces de estudio incluidas
from TTS.api import TTS
xt = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(dev)
names = list(getattr(xt, 'speakers', None) or [])
for who, frases in cfg['frases'].items():
    st = cfg['estilo'][who]; spk = cfg['xtts'][who]
    if names and spk not in names: spk = names[0]; print('aviso: voz XTTS no encontrada, uso', spk, flush=True)
    for i, t in enumerate(frases):
        p = os.path.join(T, f'xtts_{who}_{i + 1}.wav'); xt.tts_to_file(text=t, speaker=spk, language='es', file_path=p, speed=st['xtts_speed']); print('OK', p, flush=True)
'''
with Paso('2b · generar pruebas de voz'):
    cfgp = os.path.join(TMP, 'cfg.json'); json.dump({'frases': FRASES, 'estilo': ESTILO, 'tmp': TMP,
                                                    'xtts': {'confundido': VOZ_XTTS_CONFUNDIDO, 'explicador': VOZ_XTTS_EXPLICADOR}}, open(cfgp, 'w'), ensure_ascii=False)
    open(os.path.join(TMP, 'gen.py'), 'w').write(SCRIPT)
    pr = subprocess.Popen(['/content/voz_env/bin/python', os.path.join(TMP, 'gen.py'), cfgp], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in pr.stdout:
        tail = (tail + [line])[-30:]
        if line.startswith(('OK', 'aviso')): print(' ', line.strip().replace(TMP + '/', ''))
    pr.wait()
    if pr.returncode != 0: raise RuntimeError("Falló la generación:\n" + ''.join(tail[-12:]))

In [ ]:
#@title 3 · Dirección de voz (tono y velocidad), silencios recortados, duraciones reales y guardado en Drive
import numpy as np, soundfile as sf
SR = 48000
def ffmpeg_fx(src, dst, semitonos, tempo):
    """Sube/baja el tono sin cambiar la duración (y ajusta la velocidad), recorta silencios de inicio y fin."""
    k = 2 ** (semitonos / 12)
    af = (f"aresample={SR},asetrate={SR}*{k:.5f},aresample={SR},atempo={tempo / k:.5f},"
          "silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.02,areverse,"
          "silenceremove=start_periods=1:start_threshold=-45dB:start_silence=0.05,areverse,loudnorm=I=-16:TP=-1.5")
    subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', src, '-af', af, '-ar', str(SR), '-ac', '1', dst], check=True)
def dur(p): return sf.info(p).duration
rows = []
with Paso('2c · dirección de voz y duraciones'):
    for motor in ('chatterbox', 'xtts'):
        for who, frases in FRASES.items():
            st = ESTILO[who]
            for i, t in enumerate(frases):
                src = os.path.join(TMP, f'{motor}_{who}_{i + 1}.wav')
                if not os.path.exists(src): rows.append([motor, who, i + 1, t, '—', 'falta']); continue
                dst = os.path.join(OUT, f'{motor}_{who}_{i + 1}.wav')
                # XTTS ya trae la velocidad en el modelo; Chatterbox no: se la doy con atempo
                ffmpeg_fx(src, dst, st['semitonos'], st['tempo'] if motor == 'chatterbox' else 1.0)
                rows.append([motor, who, i + 1, t, f"{dur(dst):.2f}", os.path.basename(dst)])
        # secuencia de prueba con pausas de 0.3 s entre líneas, en el orden del guion
        orden = [('confundido', 0), ('explicador', 0), ('confundido', 1), ('explicador', 1), ('confundido', 2), ('explicador', 2)]
        parts = []
        for who, i in orden:
            p_ = os.path.join(OUT, f'{motor}_{who}_{i + 1}.wav')
            if os.path.exists(p_): parts += [sf.read(p_, dtype='float32')[0], np.zeros(int(0.3 * SR), np.float32)]
        if parts:
            sf.write(os.path.join(OUT, f'secuencia_{motor}_pausas_0.3s.wav'), np.concatenate(parts[:-1]), SR)
    md = ["# Paso 2 · Pruebas de voz", "", "| Motor | Personaje | # | Frase | Duración real (s) | Archivo |", "|---|---|---|---|---|---|"]
    md += [f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]} | {r[5]} |" for r in rows]
    open(os.path.join(OUT, 'duraciones.md'), 'w').write('\n'.join(md) + '\n')
    for r in rows: print(f"{r[0]:10s} {r[1]:11s} {r[2]}  {r[4]:>5s} s  {r[3]}")
try:
    from IPython.display import Audio, display, HTML
    for motor in ('chatterbox', 'xtts'):
        p_ = os.path.join(OUT, f'secuencia_{motor}_pausas_0.3s.wav')
        if os.path.exists(p_): display(HTML(f"<b>{motor}</b>")); display(Audio(p_))
except Exception: pass
print("Audios y tabla en:", OUT)
desconectar()